# multiprocessing 模块
- 在 Python 中，多进程管理主要依赖于标准库中的 multiprocessing 模块。

- 这个模块可以让你创建和管理多个进程，从而绕过全局解释器锁（GIL）的限制，充分利用多核 CPU 的性能，尤其适合处理计算密集型任务。

以下是 Python 多进程管理的核心方法：

## 🚀 创建和管理进程

- 最直接的方式是使用 multiprocessing.Process 类来创建单个进程。

    - start(): 启动进程。
    - join(): 阻塞主进程，等待子进程结束，这对于确保所有任务完成和资源正确释放至关重要。

In [1]:
from multiprocessing import Process
import os

def worker(name):
    print(f"工作进程{name}(PID:{os.getpid()})正在运行")
    
if __name__ == '__main__':
    # 创建进程对象
    p1 = Process(target=worker,args=('a',))
    p2 = Process(target=worker,args=('b',))
    
    # 启动进程
    p1.start()
    p2.start()
    
    # 等待所有进程执行完毕
    p1.join()
    p2.join()
    print('主进程结束')

主进程结束


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/tao/anaconda3/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/tao/anaconda3/lib/python3.11/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'worker' on <module '__main__' (built-in)>
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/tao/anaconda3/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/tao/anaconda3/lib/python3.11/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute 'worker' on <mo

## 使用进程池（pool）

当需要批量处理大量任务时，手动创建和管理大量进程会非常繁琐。multiprocessing.Pool 类提供了一个进程池，可以自动管理一组工作进程，是处理批量任务的首选方式。

- map(func, iterable): 将一个可迭代对象中的每个元素传递给函数 func 处理，并返回结果列表。
- apply_async(func, args, callback): 异步执行单个任务，callback 参数可以指定任务完成后执行的回调函数。
- close(): 关闭进程池，不再接受新任务。
- join(): 等待进程池中所有任务完成。

In [ ]:
from multiprocessing import Pool

def square(x):
    return x * x

if __name__ == '__main__':
    # 创建一个包含4个进程的进程池
    with Pool(processes=4) as pool:
        # 将任务列表分发给进程池中的进程并行处理
        results = pool.map(square,[1,2,3,4,5])
        print(results) # 输出: [1, 4, 9, 16, 25]

## 🔗 进程间通信 (IPC)

<font color='red'>由于每个进程拥有独立的内存空间，它们之间不能直接共享变量。multiprocessing 模块提供了多种进程间通信机制。</font>

### 队列 (Queue)
- Queue 是一种线程和进程安全的先进先出（FIFO）数据结构，非常适合“生产者-消费者”模式。


In [ ]:
from multiprocessing import Process,Queue

def producer(q):
    for i in range(3):
        q.put(i)
        print(f"生产了：{i}")
        
    q.put(None) # 发送结束信号
    
def consumer(q):
    while(True):
        item = q.get()
        if item is None: # 收到结束信号
            break
        print(f"消费了：{item}")

if __name__ == '__main__':
    q = Queue()
    p = Process(target=producer,args=(q,))
    c = Process(target=consumer,args=(q,))
    
    p.start()
    c.start()
    
    p.join()
    c.join()

### 管道 (Pipe)
- Pipe 返回一对连接对象，适合在两个进程之间进行高效的双向通信。
- 相比于 Queue，Pipe 更加底层、轻量且高效，但它的使用场景相对受限（主要用于点对点通信）。

#### 核心概念与工作原理

Pipe 的本质是操作系统提供的一个“管道”资源。当你调用 multiprocessing.Pipe() 时，<font color='red'>它会返回两个连接对象（Connection objects），分别代表管道的两端。</font>

- 全双工 (Duplex)：默认情况下（duplex=True），管道是双向的。这意味着两端都可以发送数据，也都可以接收数据。
- 半双工 (Simplex)：如果设置 duplex=False，管道是单向的。通常一端只能发送，另一端只能接收。
- 数据流向：数据在管道中是先进先出 (FIFO) 的。

#### 关键方法与属性

Connection 对象（即管道两端）提供了以下主要方法：

- send(obj): 发送一个对象。该对象必须是可序列化（picklable）的。
- recv(): 接收一个对象。如果管道中没有数据，该方法会阻塞（等待），直到有数据到来。
- poll(timeout=None): 检查管道中是否有数据可读。
    - 这是一个非阻塞操作（或者带超时阻塞），常用于避免 recv() 无限期等待。
    - 用法：if conn.poll(1): data = conn.recv()
- close(): 关闭连接。
- fileno(): 返回连接使用的文件描述符（整数），用于更底层的操作。

#### 基本用法与代码示例

- 这是最标准的使用模式：主进程创建一个管道，将其中一端传递给子进程，实现父子进程间的通信。

| 特性 | Pipe (管道) | Queue (队列) |
| :--- | :--- | :--- |
| 适用场景 | 两个进程之间的通信 (点对点) | 多个进程之间的通信 (多对多) |
| 性能 | 极高 (无锁，底层实现简单) | 较高 (内部包含锁机制，开销稍大) |
| 通信模式 | 全双工或半双工 | 先进先出 (FIFO) |
| 数据对象 | 任何可 pickle 的对象 | 任何可 pickle 的对象 |
| 复杂度 | 需要手动管理连接关闭，否则可能死锁 | 自动管理，使用更简单 |



In [ ]:
from multiprocessing import Process, Pipe

def worker(conn):
    # 子进程逻辑
    # 1. 接收数据
    data = conn.recv()
    print(f"子进程收到: {data}")
    
    # 2. 发送数据回父进程
    conn.send("任务完成！")
    conn.close() # 好习惯：通信结束后关闭连接

if __name__ == '__main__':
    # 1. 创建管道，返回两个连接对象 (parent_conn, child_conn)
    parent_conn, child_conn = Pipe()
    
    # 2. 创建进程，将 child_conn 传递给子进程
    p = Process(target=worker, args=(child_conn,))
    p.start()
    
    # 3. 父进程发送数据
    parent_conn.send("你好，子进程！")
    
    # 4. 父进程接收子进程的回复
    print(f"父进程收到: {parent_conn.recv()}")
    
    # 5. 等待进程结束并关闭父进程端的连接
    p.join()
    parent_conn.close()

#### 高级注意事项与“坑”
在使用 Pipe 时，有几个非常关键的细节需要注意，否则容易导致程序卡死（死锁）：

##### A. 必须正确关闭连接 (防止死锁)

这是新手最容易犯的错误。recv() 方法只有在管道的一端关闭，且缓冲区数据被读空后，才会抛出 EOFError 异常。
- 规则：如果子进程只使用 child_conn，那么子进程应该只保留它需要的那一端，或者在通信结束后显式调用 close()。
- 后果：如果父进程保留了 child_conn 的引用（即使它不用），子进程的 recv() 可能会一直等待，因为它认为父进程可能还会发送数据。

##### B. 处理 EOFError

当管道的一端关闭，而另一端尝试 recv() 时，会抛出 EOFError。你应该使用 try...except 块来优雅地处理这种情况。

In [ ]:
# 接收端的推荐写法
while True:
    try:
        data = conn.recv()
        # 处理数据
    except EOFError:
        # 管道已关闭，退出循环
        break

##### C. 数据序列化

和 Queue 一样，Pipe 传输的数据会被序列化（Pickle）。这意味着你不能直接传输文件句柄、数据库连接或未定义的类对象。

##### D. 避免在进程池 (Pool) 中直接使用

虽然技术上可行，但 Pipe 并不适合直接配合 multiprocessing.Pool 使用，因为进程池中的进程是复用的，管理管道生命周期比较复杂。在进程池中，通常推荐使用回调函数或 Queue。

#### 总结

Pipe 是 Python 多进程编程中最快的通信方式，但它要求开发者对资源管理（特别是 close()）有更精细的控制。对于简单的“一问一答”或“流式传输”场景，它是最佳选择。

### 管理器 (Manager)
- Manager 可以创建一个共享的服务器进程，其他进程可以通过代理对象来操作共享数据（如列表、字典），非常灵活。

In [ ]:
from multiprocessing import Manager

def modify_shared_data(shared_list, shared_dict):
    shared_list.append(100)
    shared_dict['a'] = 'hello'

if __name__ == '__main__':
    with Manager() as manager:
        # 创建可在进程间共享的列表和字典
        my_list = manager.list()
        my_dict = manager.dict()
        
        p = Process(target=modify_shared_data, args=(my_list, my_dict))
        p.start()
        p.join()
        
        print(my_list)  # 输出: [100]
        print(my_dict)  # 输出: {'a': 'hello'}

### 🔒 进程同步

当多个进程需要访问共享资源时，需要使用同步原语来避免数据竞争。

- 锁 (Lock): 确保同一时间只有一个进程可以执行某段代码（临界区）。
- 信号量 (Semaphore): 控制同时访问某个资源的进程数量。

### ⚙️ 守护进程 (Daemon Process)
- 守护进程是一种在后台运行的进程，当主进程退出时，它会自动被终止，无需显式等待其结束。

In [ ]:
from multiprocessing import Process
import time

def background_task():
    while True:
        print("后台任务运行中...")
        time.sleep(1)

if __name__ == '__main__':
    p = Process(target=background_task, daemon=True) # 设置为守护进程
    p.start()
    print("主进程即将退出...")
    # 主进程退出后，守护进程 p 也会被自动终止

| 特性 | 多进程 (Multiprocessing) | 多线程 (Multithreading) |
| :--- | :--- | :--- |
| 适用场景 | CPU 密集型任务 (如科学计算、图像处理) | I/O 密集型任务 (如网络请求、文件读写) |
| GIL 影响 | 可以绕过 GIL，实现真正的并行计算 | 受 GIL 限制，无法利用多核并行 |
| 内存与通信 | 内存独立，通信开销较大，需使用 IPC | 共享内存，通信简单，但需注意线程安全 |
| 创建开销 | 开销较大，创建和销毁较慢 | 开销较小，创建和切换更快 |
[(video_note_list_1)]